# Viraltest v2 — TRL GRPO Training

Train Qwen2.5-1.5B-Instruct on the Viraltest environment using Group Relative Policy Optimization.

**Requirements:** Free Colab T4 GPU, ~30 min for 100 episodes.

**Reward:** per-step env reward (0-1) + 2× terminal grader_score.

In [ ]:
!pip install -q trl transformers accelerate peft bitsandbytes openai httpx matplotlib

In [ ]:
import json
import os
import matplotlib.pyplot as plt
from typing import List, Dict, Any

# Set your env server URL (run the Docker container or HF Space first)
ENV_BASE_URL = os.getenv("ENV_BASE_URL", "http://localhost:8000")
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Environment: {ENV_BASE_URL}")
print(f"Model: {MODEL_NAME}")

## Episode Collection

Run the agent against the environment and collect (prompt, response, reward) tuples.

In [ ]:
import httpx

def reset_env(task: str = "monthly_engage") -> Dict[str, Any]:
    resp = httpx.post(f"{ENV_BASE_URL}/reset", json={"task": task}, timeout=30)
    return resp.json()

def step_env(action: Dict[str, Any]) -> Dict[str, Any]:
    resp = httpx.post(f"{ENV_BASE_URL}/step", json=action, timeout=30)
    return resp.json()

def collect_episode(task: str, max_steps: int = 30) -> List[Dict[str, Any]]:
    """Collect one episode of (obs, action, reward) tuples."""
    obs = reset_env(task)
    trajectory = []
    for step in range(max_steps):
        obs_data = obs.get("observation", {})
        if obs.get("done", False):
            break
        # Simple heuristic agent for data collection
        action = {
            "scheduled_actions": [
                {"hour": 12, "action_type": "post", "content_type": "carousel",
                 "topic": "AI tools", "tags": ["ai", "coding"], "intent": "save_bait"},
            ],
            "notes": f"Step {step}: collecting training data."
        }
        obs = step_env(action)
        reward = obs.get("reward", 0.0)
        trajectory.append({"obs": obs_data, "action": action, "reward": reward})
    return trajectory

# Collect baseline episodes
print("Collecting baseline episodes...")
baseline_rewards = []
for task in ["monthly_engage", "monthly_strategic", "monthly_competitive"]:
    traj = collect_episode(task)
    total_reward = sum(t["reward"] for t in traj)
    baseline_rewards.append(total_reward)
    print(f"  {task}: {total_reward:.4f} ({len(traj)} steps)")

## GRPO Training Loop

Uses TRL's GRPOTrainer with the environment reward as the RL signal.

In [ ]:
# NOTE: Full GRPO training requires:
# 1. Running the env server (docker or uvicorn)
# 2. A reward function that maps env observations to scalar rewards
# 3. Enough GPU memory for the model + optimizer
#
# This skeleton shows the structure. Adapt based on your compute.

from transformers import AutoTokenizer, AutoModelForCausalLM
# from trl import GRPOConfig, GRPOTrainer  # uncomment when running

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype="auto")

print(f"Tokenizer loaded: {MODEL_NAME}")
print("To run full training, uncomment model loading and GRPOTrainer setup.")

## Plot Reward Curves

In [ ]:
# Placeholder — replace with actual training rewards
import numpy as np

episodes = list(range(1, 201))
# Simulated reward curve (replace with real data)
rewards = np.cumsum(np.random.randn(200) * 0.02 + 0.01)
rewards = np.clip(rewards, 0, 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episodes, rewards, linewidth=1.5, color='#2196F3')
ax.set_xlabel('Episode')
ax.set_ylabel('Cumulative Reward')
ax.set_title('Viraltest v2 — GRPO Training Reward Curve')
ax.grid(True, alpha=0.3)
fig.savefig('../plots/reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/reward_curve.png')

In [ ]:
# Before vs After comparison
tasks = ['monthly_engage', 'monthly_strategic', 'monthly_competitive']
before_scores = [0.12, 0.10, 0.08]  # Replace with actual baseline
after_scores = [0.45, 0.35, 0.28]   # Replace with actual trained

x = np.arange(len(tasks))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, before_scores, width, label='Baseline', color='#FF9800')
bars2 = ax.bar(x + width/2, after_scores, width, label='Trained (GRPO)', color='#4CAF50')

ax.set_ylabel('Grader Score')
ax.set_title('Before vs After Training — Grader Scores')
ax.set_xticks(x)
ax.set_xticklabels(tasks, rotation=15)
ax.legend()
ax.set_ylim(0, 0.8)
ax.grid(True, alpha=0.3, axis='y')

fig.savefig('../plots/before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/before_after.png')